# [common configs, imports etc]

In [11]:
import numpy as np

# regression MLP

## "simplest possible" regression MLP - Net

In [ ]:
class Net:
    def __init__(self, *, x_sz, y1_sz, y2_sz):
        self.W1 = np.random.randn(x_sz, y1_sz)  # W1 ~ (x_sz, y1_sz)
        print(f"W1: {self.W1.dtype} {self.W1.shape}")

        self.W2 = np.random.randn(y1_sz, y2_sz)  # W2 ~ (y1_sz, y2_sz)
        print(f"W2: {self.W2.dtype} {self.W2.shape}")

    @staticmethod
    def _f1(A):  # ReLU
        return A * (A > 0)

    @staticmethod
    def _df1(A):  # dReLU
        return (A > 0).astype(A.dtype)

    @staticmethod
    def _f2(A):  # linear / identity
        return A

    @staticmethod
    def _df2(A):  # d(linear) == 1
        return np.ones_like(A)

    def forward(self, X):
        self.X = X  # X ~ (b_sz, x_sz)
        self.Z1 = self.X @ self.W1  # Z1 ~ (b_sz, y1_sz)
        self.Y1 = self._f1(self.Z1)  # Y1 ~ (b_sz, y1_sz)
        self.Z2 = self.Y1 @ self.W2  # Z2 ~ (b_sz, y2_sz)
        self.Y2 = self._f2(self.Z2)  # Y2 ~ (b_sz, y2_sz)
        return self.Y2

    @staticmethod
    def _loss(Y_predicted, Y_true):  # 1/(2N) * Sum (y-y_true)^2, mean over batch
        return 0.5 * ((Y_predicted - Y_true) ** 2).sum() / Y_predicted.shape[0]

    @staticmethod
    def _dloss(Y_predicted, Y_true):
        return (Y_predicted - Y_true) / Y_predicted.shape[0]

    def backward(self, Y_true, lr=1e-3):
        # E2 ~ (b_sz, y2_sz)
        self.E2 = self._dloss(self.Y2, Y_true) * self._df2(self.Z2)

        # E1 ~ (b_sz, y1_sz)
        self.E1 = self.E2 @ self.W2.T * self._df1(self.Z1)

        # dL_dW2 ~ (y1_sz, y2_sz)
        self.dL_dW2 = self.Y1.T @ self.E2

        # dL_dW1 ~ (x_sz, y1_sz)
        self.dL_dW1 = self.X.T @ self.E1

        self.W2 += -lr * self.dL_dW2
        self.W1 += -lr * self.dL_dW1


nn = Net(x_sz=5, y1_sz=16, y2_sz=2)

W1: float64 (5, 16)
W2: float64 (16, 2)


## regression MLP with bias - NetB

In [ ]:
# `Net` + CLASSIC explicit bias: every layer keeps a separate bias VECTOR (b1, b2) and
# computes  Z = X @ W + b.  This is the textbook formulation.
#   - Contrast with `Net`, which is meant to be used with the "augmentation trick"
#     instead: the caller appends a column of ones to the input and the bias lives in an
#     extra weight row -- mathematically equivalent, with no explicit b parameter.
#   - Bias is added at EVERY layer (both hidden and output) -- see the chat note on when
#     per-layer bias is/ isn't worthwhile.
class NetB:
    def __init__(self, *, x_sz, y1_sz, y2_sz):
        self.W1 = np.random.randn(x_sz, y1_sz)  # W1 ~ (x_sz, y1_sz)
        print(f"W1: {self.W1.dtype} {self.W1.shape}")
        self.b1 = np.zeros(y1_sz)  # b1 ~ (y1_sz,)   (classic bias, zero-init)
        print(f"b1: {self.b1.dtype} {self.b1.shape}")

        self.W2 = np.random.randn(y1_sz, y2_sz)  # W2 ~ (y1_sz, y2_sz)
        print(f"W2: {self.W2.dtype} {self.W2.shape}")
        self.b2 = np.zeros(y2_sz)  # b2 ~ (y2_sz,)   (classic bias, zero-init)
        print(f"b2: {self.b2.dtype} {self.b2.shape}")

    @staticmethod
    def _f1(A):  # ReLU
        return A * (A > 0)

    @staticmethod
    def _df1(A):  # dReLU
        return (A > 0).astype(A.dtype)

    @staticmethod
    def _f2(A):  # linear / identity
        return A

    @staticmethod
    def _df2(A):  # d(linear) == 1
        return np.ones_like(A)

    def forward(self, X):
        self.X = X  # X ~ (b_sz, x_sz)
        self.Z1 = (
            self.X @ self.W1 + self.b1
        )  # Z1 ~ (b_sz, y1_sz)   (+ b1 broadcasts over batch)
        self.Y1 = self._f1(self.Z1)  # Y1 ~ (b_sz, y1_sz)
        self.Z2 = (
            self.Y1 @ self.W2 + self.b2
        )  # Z2 ~ (b_sz, y2_sz)   (+ b2 broadcasts over batch)
        self.Y2 = self._f2(self.Z2)  # Y2 ~ (b_sz, y2_sz)
        return self.Y2

    @staticmethod
    def _loss(Y_predicted, Y_true):  # 1/(2N) * Sum (y-y_true)^2, mean over batch
        return 0.5 * ((Y_predicted - Y_true) ** 2).sum() / Y_predicted.shape[0]

    @staticmethod
    def _dloss(Y_predicted, Y_true):
        return (Y_predicted - Y_true) / Y_predicted.shape[0]

    def backward(self, Y_true, lr=1e-3):
        # E2 ~ (b_sz, y2_sz)
        self.E2 = self._dloss(self.Y2, Y_true) * self._df2(self.Z2)

        # E1 ~ (b_sz, y1_sz)
        self.E1 = self.E2 @ self.W2.T * self._df1(self.Z1)

        # dL_dW2 ~ (y1_sz, y2_sz)
        self.dL_dW2 = self.Y1.T @ self.E2
        # dL_db2 ~ (y2_sz,)   (bias gradient = error summed over the batch)
        self.dL_db2 = self.E2.sum(axis=0)

        # dL_dW1 ~ (x_sz, y1_sz)
        self.dL_dW1 = self.X.T @ self.E1
        # dL_db1 ~ (y1_sz,)   (bias gradient = error summed over the batch)
        self.dL_db1 = self.E1.sum(axis=0)

        self.W2 += -lr * self.dL_dW2
        self.b2 += -lr * self.dL_db2
        self.W1 += -lr * self.dL_dW1
        self.b1 += -lr * self.dL_db1


nn_b = NetB(x_sz=5, y1_sz=16, y2_sz=2)

W1: float64 (5, 16)
b1: float64 (16,)
W2: float64 (16, 2)
b2: float64 (2,)


## fixed-memory regression MLP - NetFixedMem

In [ ]:
# Fixed-memory version of `Net`: ALL arrays are allocated ONCE in __init__, and the
# forward/backward hot path only writes into those preallocated buffers in place
# (no per-pass allocation). This mirrors how you'd write it in plain C: malloc every
# matrix up front, then each pass is just BLAS gemm calls with a preallocated output
# plus elementwise loops.
#
# Consequences / how it differs from Net:
#   - The batch size is FIXED at construction (buffers are sized (b_sz, ...)). Inputs
#     must be exactly (b_sz, x_sz); `forward` copies them into the preallocated X buffer.
#   - The big matmuls write into preallocated buffers via np.dot(A, B, out=C),
#     i.e. C = A @ B (a BLAS gemm into fixed memory, the C analogue).
#   - Activations are GENERIC, swappable AND allocation-free: each of _f1/_df1/_f2/_df2
#     takes an optional `out` buffer -- THE blueprint pattern, since `out` is just a
#     destination pointer in C. Three call modes from one signature:
#         f(A)           -> returns a NEW array            (simple/educational, allocates)
#         f(A, out=dst)  -> writes into dst, returns dst   (no alloc; dst may differ from A)
#         f(A, out=A)    -> modifies A IN PLACE            (no alloc, no extra buffer)
#     forward writes activations straight into Y1/Y2; backward writes activation
#     derivatives into the scratch buffers D1/D2. So the only thing C would still need
#     to add is manual malloc of these same buffers up front (already done in __init__).
#   - The SGD step scales the gradient buffers in place by lr (a "fused" update), so
#     after backward() dL_dW* hold lr*grad rather than the raw grad -- fine for training,
#     and it avoids allocating a `lr * grad` temporary.
#   - `_loss` (diagnostic) and `_dloss` still return a fresh array; giving them the same
#     `out=` treatment is a trivial further exercise.
class NetFixedMem:
    def __init__(self, *, b_sz, x_sz, y1_sz, y2_sz):
        # --- parameters (weights) ---
        self.W1 = np.random.randn(x_sz, y1_sz)  # W1 ~ (x_sz, y1_sz)
        print(f"W1: {self.W1.dtype} {self.W1.shape}")
        self.W2 = np.random.randn(y1_sz, y2_sz)  # W2 ~ (y1_sz, y2_sz)
        print(f"W2: {self.W2.dtype} {self.W2.shape}")

        # --- preallocated forward buffers ---
        # X ~ (b_sz, x_sz)
        self.X = np.zeros((b_sz, x_sz))  # input copied here each forward
        self.Z1 = np.zeros((b_sz, y1_sz))  # Z1 ~ (b_sz, y1_sz)
        self.Y1 = np.zeros((b_sz, y1_sz))  # Y1 ~ (b_sz, y1_sz)
        self.Z2 = np.zeros((b_sz, y2_sz))  # Z2 ~ (b_sz, y2_sz)
        self.Y2 = np.zeros((b_sz, y2_sz))  # Y2 ~ (b_sz, y2_sz)

        # --- preallocated backward buffers ---
        # E2 ~ (b_sz, y2_sz)
        self.E2 = np.zeros((b_sz, y2_sz))  # dL/dZ2
        # E1 ~ (b_sz, y1_sz)
        self.E1 = np.zeros((b_sz, y1_sz))  # dL/dZ1
        # D1, D2: scratch for activation derivatives f1'(Z1), f2'(Z2)
        self.D1 = np.zeros((b_sz, y1_sz))  # D1 ~ (b_sz, y1_sz)
        self.D2 = np.zeros((b_sz, y2_sz))  # D2 ~ (b_sz, y2_sz)
        self.dL_dW2 = np.zeros((y1_sz, y2_sz))  # dL_dW2 ~ (y1_sz, y2_sz)
        self.dL_dW1 = np.zeros((x_sz, y1_sz))  # dL_dW1 ~ (x_sz, y1_sz)
        for name in (
            "X",
            "Z1",
            "Y1",
            "Z2",
            "Y2",
            "E2",
            "E1",
            "D1",
            "D2",
            "dL_dW1",
            "dL_dW2",
        ):
            a = getattr(self, name)
            print(f"{name}: {a.dtype} {a.shape}")

    # --- generic, swappable, allocation-free activations ---
    # Each takes an optional `out` buffer (== a destination pointer in C):
    #   out=None  -> return a NEW array      out=dst -> write into dst      out=A -> in place
    @staticmethod
    def _f1(A, out=None):  # ReLU
        return np.maximum(A, 0.0, out=out)

    @staticmethod
    def _df1(A, out=None):  # dReLU: 1.0 where A > 0 else 0.0
        # heaviside(x, h) = step fn:
        #   0 if x<0,
        #   h if x==0,
        #   1 if x>0
        # (here h=0 -> f'(0)=0)
        return np.heaviside(A, 0.0, out=out)

    @staticmethod
    def _f2(A, out=None):  # linear / identity
        if out is None:
            return A
        np.copyto(out, A)
        return out

    @staticmethod
    def _df2(A, out=None):  # d(linear) == 1
        if out is None:
            return np.ones_like(A)
        out[:] = 1.0
        return out

    @staticmethod
    def _loss(Y_predicted, Y_true):  # diagnostic only (allocates a small temp)
        return 0.5 * ((Y_predicted - Y_true) ** 2).sum() / Y_predicted.shape[0]

    @staticmethod
    def _dloss(Y_predicted, Y_true):
        return (Y_predicted - Y_true) / Y_predicted.shape[0]

    def forward(self, X):
        # X ~ (b_sz, x_sz)
        np.copyto(self.X, X)  # X buffer <- input (fixed batch)
        # Z1 ~ (b_sz, y1_sz)
        np.dot(self.X, self.W1, out=self.Z1)  # Z1 = X @ W1   (preallocated out)
        # Y1 ~ (b_sz, y1_sz)
        self._f1(self.Z1, out=self.Y1)  # Y1 = f1(Z1)   (alloc-free; Z1 preserved)
        # Z2 ~ (b_sz, y2_sz)
        np.dot(self.Y1, self.W2, out=self.Z2)  # Z2 = Y1 @ W2  (preallocated out)
        # Y2 ~ (b_sz, y2_sz)
        self._f2(self.Z2, out=self.Y2)  # Y2 = f2(Z2)   (alloc-free)
        return self.Y2

    def backward(self, Y_true, lr=1e-3):
        # E2 = dL/dZ2 = dloss(Y2, Y_true) * f2'(Z2)   (generic, identical to Net)
        # E2 ~ (b_sz, y2_sz)
        self.E2[:] = self._dloss(
            self.Y2, Y_true
        )  # E2 = dloss(Y2, Y_true)  (small temp)
        self._df2(self.Z2, out=self.D2)  # D2 = f2'(Z2)  (alloc-free into scratch)
        self.E2 *= self.D2  # E2 *= f2'(Z2)

        # E1 = (E2 @ W2.T) * f1'(Z1)
        # E1 ~ (b_sz, y1_sz)
        np.dot(self.E2, self.W2.T, out=self.E1)  # E1 = E2 @ W2.T  (preallocated out)
        self._df1(self.Z1, out=self.D1)  # D1 = f1'(Z1)  (alloc-free into scratch)
        self.E1 *= self.D1  # E1 *= f1'(Z1)

        # weight gradients (preallocated out)
        # dL_dW2 ~ (y1_sz, y2_sz)
        np.dot(self.Y1.T, self.E2, out=self.dL_dW2)  # dL_dW2 = Y1.T @ E2
        # dL_dW1 ~ (x_sz, y1_sz)
        np.dot(self.X.T, self.E1, out=self.dL_dW1)  # dL_dW1 = X.T  @ E1

        # fused SGD step: scale grads in place by lr, then subtract (no temp alloc)
        self.dL_dW2 *= lr
        self.dL_dW1 *= lr
        self.W2 -= self.dL_dW2
        self.W1 -= self.dL_dW1


nn_fm = NetFixedMem(b_sz=8, x_sz=5, y1_sz=16, y2_sz=2)

W1: float64 (5, 16)
W2: float64 (16, 2)
X: float64 (8, 5)
Z1: float64 (8, 16)
Y1: float64 (8, 16)
Z2: float64 (8, 2)
Y2: float64 (8, 2)
E2: float64 (8, 2)
E1: float64 (8, 16)
D1: float64 (8, 16)
D2: float64 (8, 2)
dL_dW1: float64 (5, 16)
dL_dW2: float64 (16, 2)


## fixed-memory regression MLP with bias - NetFixedMemB

In [ ]:
# `NetFixedMem` + CLASSIC explicit bias (the same bias that NetB adds to Net).
# This is intentionally a near-verbatim copy of NetFixedMem; EVERY bias-specific line is
# marked with a `(bias)` comment so the diff is obvious. The fixed-memory rationale
# (preallocate once, write in place, np.dot(out=), alloc-free activations, fused SGD)
# is unchanged -- see NetFixedMem for those details.
#
# Bias additions, all staying within the preallocate-once discipline:
#   - b1, b2: bias VECTORS, allocated once (zero-init).
#   - dL_db1, dL_db2: their gradient buffers, allocated once.
#   - forward: after each matmul, add the bias in place (Z += b broadcasts over batch).
#   - backward: bias gradient = error summed over the batch, written into its buffer via
#     np.sum(..., out=...); then folded into the same fused SGD step as the weights.
class NetFixedMemB:
    def __init__(self, *, b_sz, x_sz, y1_sz, y2_sz):
        # --- parameters (weights + biases) ---
        self.W1 = np.random.randn(x_sz, y1_sz)  # W1 ~ (x_sz, y1_sz)
        print(f"W1: {self.W1.dtype} {self.W1.shape}")
        self.b1 = np.zeros(
            y1_sz
        )  # (bias) b1 ~ (y1_sz,)   classic bias vector, zero-init
        print(f"b1: {self.b1.dtype} {self.b1.shape}")
        self.W2 = np.random.randn(y1_sz, y2_sz)  # W2 ~ (y1_sz, y2_sz)
        print(f"W2: {self.W2.dtype} {self.W2.shape}")
        self.b2 = np.zeros(
            y2_sz
        )  # (bias) b2 ~ (y2_sz,)   classic bias vector, zero-init
        print(f"b2: {self.b2.dtype} {self.b2.shape}")

        # --- preallocated forward buffers ---
        # X ~ (b_sz, x_sz)
        self.X = np.zeros((b_sz, x_sz))  # input copied here each forward
        self.Z1 = np.zeros((b_sz, y1_sz))  # Z1 ~ (b_sz, y1_sz)
        self.Y1 = np.zeros((b_sz, y1_sz))  # Y1 ~ (b_sz, y1_sz)
        self.Z2 = np.zeros((b_sz, y2_sz))  # Z2 ~ (b_sz, y2_sz)
        self.Y2 = np.zeros((b_sz, y2_sz))  # Y2 ~ (b_sz, y2_sz)

        # --- preallocated backward buffers ---
        # E2 ~ (b_sz, y2_sz)
        self.E2 = np.zeros((b_sz, y2_sz))  # dL/dZ2
        # E1 ~ (b_sz, y1_sz)
        self.E1 = np.zeros((b_sz, y1_sz))  # dL/dZ1
        # D1, D2: scratch for activation derivatives f1'(Z1), f2'(Z2)
        self.D1 = np.zeros((b_sz, y1_sz))  # D1 ~ (b_sz, y1_sz)
        self.D2 = np.zeros((b_sz, y2_sz))  # D2 ~ (b_sz, y2_sz)
        self.dL_dW2 = np.zeros((y1_sz, y2_sz))  # dL_dW2 ~ (y1_sz, y2_sz)
        self.dL_dW1 = np.zeros((x_sz, y1_sz))  # dL_dW1 ~ (x_sz, y1_sz)
        # (bias) preallocated bias-gradient buffers
        self.dL_db2 = np.zeros(y2_sz)  # (bias) dL_db2 ~ (y2_sz,)
        self.dL_db1 = np.zeros(y1_sz)  # (bias) dL_db1 ~ (y1_sz,)
        for name in (
            "X",
            "Z1",
            "Y1",
            "Z2",
            "Y2",
            "E2",
            "E1",
            "D1",
            "D2",
            "dL_dW1",
            "dL_dW2",
            "dL_db1",  # (bias)
            "dL_db2",  # (bias)
        ):
            a = getattr(self, name)
            print(f"{name}: {a.dtype} {a.shape}")

    # --- generic, swappable, allocation-free activations ---
    # Each takes an optional `out` buffer (== a destination pointer in C):
    #   out=None  -> return a NEW array      out=dst -> write into dst      out=A -> in place
    @staticmethod
    def _f1(A, out=None):  # ReLU
        return np.maximum(A, 0.0, out=out)

    @staticmethod
    def _df1(A, out=None):  # dReLU: 1.0 where A > 0 else 0.0
        # heaviside(x, h) = step fn:
        #   0 if x<0,
        #   h if x==0,
        #   1 if x>0
        # (here h=0 -> f'(0)=0)
        return np.heaviside(A, 0.0, out=out)

    @staticmethod
    def _f2(A, out=None):  # linear / identity
        if out is None:
            return A
        np.copyto(out, A)
        return out

    @staticmethod
    def _df2(A, out=None):  # d(linear) == 1
        if out is None:
            return np.ones_like(A)
        out[:] = 1.0
        return out

    @staticmethod
    def _loss(Y_predicted, Y_true):  # diagnostic only (allocates a small temp)
        return 0.5 * ((Y_predicted - Y_true) ** 2).sum() / Y_predicted.shape[0]

    @staticmethod
    def _dloss(Y_predicted, Y_true):
        return (Y_predicted - Y_true) / Y_predicted.shape[0]

    def forward(self, X):
        # X ~ (b_sz, x_sz)
        np.copyto(self.X, X)  # X buffer <- input (fixed batch)
        # Z1 ~ (b_sz, y1_sz)
        np.dot(self.X, self.W1, out=self.Z1)  # Z1 = X @ W1   (preallocated out)
        self.Z1 += self.b1  # (bias) Z1 += b1   (in-place, broadcasts over batch)
        # Y1 ~ (b_sz, y1_sz)
        self._f1(self.Z1, out=self.Y1)  # Y1 = f1(Z1)   (alloc-free; Z1 preserved)
        # Z2 ~ (b_sz, y2_sz)
        np.dot(self.Y1, self.W2, out=self.Z2)  # Z2 = Y1 @ W2  (preallocated out)
        self.Z2 += self.b2  # (bias) Z2 += b2   (in-place, broadcasts over batch)
        # Y2 ~ (b_sz, y2_sz)
        self._f2(self.Z2, out=self.Y2)  # Y2 = f2(Z2)   (alloc-free)
        return self.Y2

    def backward(self, Y_true, lr=1e-3):
        # E2 = dL/dZ2 = dloss(Y2, Y_true) * f2'(Z2)   (generic, identical to Net)
        # E2 ~ (b_sz, y2_sz)
        self.E2[:] = self._dloss(
            self.Y2, Y_true
        )  # E2 = dloss(Y2, Y_true)  (small temp)
        self._df2(self.Z2, out=self.D2)  # D2 = f2'(Z2)  (alloc-free into scratch)
        self.E2 *= self.D2  # E2 *= f2'(Z2)

        # E1 = (E2 @ W2.T) * f1'(Z1)
        # E1 ~ (b_sz, y1_sz)
        np.dot(self.E2, self.W2.T, out=self.E1)  # E1 = E2 @ W2.T  (preallocated out)
        self._df1(self.Z1, out=self.D1)  # D1 = f1'(Z1)  (alloc-free into scratch)
        self.E1 *= self.D1  # E1 *= f1'(Z1)

        # weight gradients (preallocated out)
        # dL_dW2 ~ (y1_sz, y2_sz)
        np.dot(self.Y1.T, self.E2, out=self.dL_dW2)  # dL_dW2 = Y1.T @ E2
        # (bias) dL_db2 = E2 summed over the batch  (np.sum into preallocated buffer)
        np.sum(self.E2, axis=0, out=self.dL_db2)
        # dL_dW1 ~ (x_sz, y1_sz)
        np.dot(self.X.T, self.E1, out=self.dL_dW1)  # dL_dW1 = X.T  @ E1
        # (bias) dL_db1 = E1 summed over the batch
        np.sum(self.E1, axis=0, out=self.dL_db1)

        # fused SGD step: scale grads in place by lr, then subtract (no temp alloc)
        self.dL_dW2 *= lr
        self.dL_dW1 *= lr
        self.dL_db2 *= lr  # (bias)
        self.dL_db1 *= lr  # (bias)
        self.W2 -= self.dL_dW2
        self.W1 -= self.dL_dW1
        self.b2 -= self.dL_db2  # (bias)
        self.b1 -= self.dL_db1  # (bias)


nn_fm_b = NetFixedMemB(b_sz=8, x_sz=5, y1_sz=16, y2_sz=2)

W1: float64 (5, 16)
b1: float64 (16,)
W2: float64 (16, 2)
b2: float64 (2,)
X: float64 (8, 5)
Z1: float64 (8, 16)
Y1: float64 (8, 16)
Z2: float64 (8, 2)
Y2: float64 (8, 2)
E2: float64 (8, 2)
E1: float64 (8, 16)
D1: float64 (8, 16)
D2: float64 (8, 2)
dL_dW1: float64 (5, 16)
dL_dW2: float64 (16, 2)
dL_db1: float64 (16,)
dL_db2: float64 (2,)


# classification MLP

## "simplest possible" classification MLP - DigitsRecognizerNet

In [54]:
# Closest possible adaptation of `Net` to MNIST digit classification.
#
# What CHANGES vs. Net (and why):
#   - sizes: x_sz=784 (28x28 pixels), y2_sz=10 (one score per digit class).
#   - _f2: linear -> softmax, so the 10 outputs form a probability distribution.
#   - _loss: mean squared error -> mean cross-entropy (the natural classification loss).
#   - weight init: scaled by 1/sqrt(fan_in) (He/Xavier) instead of plain randn,
#     otherwise 784 inputs make the pre-activations huge and softmax saturates.
#
# What STAYS IDENTICAL to Net (the elegant part):
#   - _f1/_df1 (ReLU), forward(), backward().
#   - _dloss returns (Y_predicted - Y_true) / N, and _df2 returns ones, EXACTLY as
#     in Net. For softmax + cross-entropy the output error dL/dZ2 collapses to
#     (softmax - one_hot)/N == (Y2 - Y_true)/N -- the same form as linear + MSE.
#     So we fold the softmax/CE derivative into this pair and reuse backward() verbatim.
#
# Y_true is expected one-hot: shape (batch, 10). Use one_hot(labels) to build it.
class DigitsRecognizerNet:
    def __init__(self, *, x_sz, y1_sz, y2_sz):
        # W1 ~ (x_sz, y1_sz)
        self.W1 = np.random.randn(x_sz, y1_sz) * np.sqrt(2.0 / x_sz)  # He init (ReLU)
        print(f"W1: {self.W1.dtype} {self.W1.shape}")

        # W2 ~ (y1_sz, y2_sz)
        self.W2 = np.random.randn(y1_sz, y2_sz) * np.sqrt(
            1.0 / y1_sz
        )  # Xavier-ish (softmax)
        print(f"W2: {self.W2.dtype} {self.W2.shape}")

    @staticmethod
    def _f1(A):  # ReLU
        return A * (A > 0)

    @staticmethod
    def _df1(A):  # dReLU
        return (A > 0).astype(A.dtype)

    @staticmethod
    def _f2(A):  # softmax (row-wise, numerically stable)
        # subtract the row-wise max for numerical stability
        # (fine bc doftmax is shift-invariant: subtracting any constant c from a row's logits gives the identical result )
        A = A - A.max(axis=1, keepdims=True)

        E = np.exp(A)

        return E / E.sum(axis=1, keepdims=True)

    @staticmethod
    def _df2(A):  # folded into _dloss for softmax+CE -> identity
        return np.ones_like(A)

    def forward(self, X):
        self.X = X  # X ~ (b_sz, x_sz)
        self.Z1 = self.X @ self.W1  # Z1 ~ (b_sz, y1_sz)
        self.Y1 = self._f1(self.Z1)  # Y1 ~ (b_sz, y1_sz)
        self.Z2 = self.Y1 @ self.W2  # Z2 ~ (b_sz, y2_sz)
        self.Y2 = self._f2(self.Z2)  # Y2 ~ (b_sz, y2_sz)
        return self.Y2

    @staticmethod
    def _loss(Y_predicted, Y_true):  # mean cross-entropy over batch
        eps = 1e-12
        return -(Y_true * np.log(Y_predicted + eps)).sum() / Y_predicted.shape[0]

    @staticmethod
    def _dloss(Y_predicted, Y_true):  # softmax+CE: dL/dZ2 == (Y2 - Y_true)/N
        return (Y_predicted - Y_true) / Y_predicted.shape[0]

    def backward(self, Y_true, lr=1e-3):
        # E2 ~ (b_sz, y2_sz)
        self.E2 = self._dloss(self.Y2, Y_true) * self._df2(self.Z2)

        # E1 ~ (b_sz, y1_sz)
        self.E1 = self.E2 @ self.W2.T * self._df1(self.Z1)

        # dL_dW2 ~ (y1_sz, y2_sz)
        self.dL_dW2 = self.Y1.T @ self.E2

        # dL_dW1 ~ (x_sz, y1_sz)
        self.dL_dW1 = self.X.T @ self.E1

        self.W2 += -lr * self.dL_dW2
        self.W1 += -lr * self.dL_dW1

    @staticmethod
    def one_hot(labels, n_classes=10):  # integer labels (batch,) -> (batch, n_classes)
        Y = np.zeros((labels.shape[0], n_classes))
        Y[np.arange(labels.shape[0]), labels] = 1.0
        return Y

    def predict(self, X):  # class index with highest probability
        return self.forward(X).argmax(axis=1)


dnn = DigitsRecognizerNet(x_sz=784, y1_sz=128, y2_sz=10)

W1: float64 (784, 128)
W2: float64 (128, 10)


## classification MLP with bias - DigitsRecognizerNetB

In [ ]:
# `DigitsRecognizerNetB` = `DigitsRecognizerNet` + classic bias, exactly the same step
# `NetB` takes from `Net`: add a bias vector at every layer and learn it.
#   - forward computes  Z = X @ W + b  (b broadcasts over the batch).
#   - backward adds the bias gradient  dL_db = error summed over the batch.
#   - `DigitsRecognizerNet` (no bias) is meant for the augmentation trick (extra column of
#     ones on the input); this version carries explicit b1/b2 instead.
# Everything else (softmax _f2, mean cross-entropy _loss, He/Xavier init, one_hot, predict)
# is identical to `DigitsRecognizerNet`.
#
# Y_true is expected one-hot: shape (batch, 10). Use one_hot(labels) to build it.
class DigitsRecognizerNetB:
    def __init__(self, *, x_sz, y1_sz, y2_sz):
        # W1 ~ (x_sz, y1_sz)
        self.W1 = np.random.randn(x_sz, y1_sz) * np.sqrt(2.0 / x_sz)  # He init (ReLU)
        print(f"W1: {self.W1.dtype} {self.W1.shape}")
        self.b1 = np.zeros(y1_sz)  # b1 ~ (y1_sz,)   (classic bias, zero-init)
        print(f"b1: {self.b1.dtype} {self.b1.shape}")

        # W2 ~ (y1_sz, y2_sz)
        self.W2 = np.random.randn(y1_sz, y2_sz) * np.sqrt(
            1.0 / y1_sz
        )  # Xavier-ish (softmax)
        print(f"W2: {self.W2.dtype} {self.W2.shape}")
        self.b2 = np.zeros(y2_sz)  # b2 ~ (y2_sz,)   (classic bias, zero-init)
        print(f"b2: {self.b2.dtype} {self.b2.shape}")

    @staticmethod
    def _f1(A):  # ReLU
        return A * (A > 0)

    @staticmethod
    def _df1(A):  # dReLU
        return (A > 0).astype(A.dtype)

    @staticmethod
    def _f2(A):  # softmax (row-wise, numerically stable)
        # subtract the row-wise max for numerical stability
        # (fine bc softmax is shift-invariant: subtracting any constant c from a row's logits gives the identical result)
        A = A - A.max(axis=1, keepdims=True)

        E = np.exp(A)

        return E / E.sum(axis=1, keepdims=True)

    @staticmethod
    def _df2(A):  # folded into _dloss for softmax+CE -> identity
        return np.ones_like(A)

    def forward(self, X):
        self.X = X  # X ~ (b_sz, x_sz)
        self.Z1 = (
            self.X @ self.W1 + self.b1
        )  # Z1 ~ (b_sz, y1_sz)   (+ b1 broadcasts over batch)
        self.Y1 = self._f1(self.Z1)  # Y1 ~ (b_sz, y1_sz)
        self.Z2 = (
            self.Y1 @ self.W2 + self.b2
        )  # Z2 ~ (b_sz, y2_sz)   (+ b2 broadcasts over batch)
        self.Y2 = self._f2(self.Z2)  # Y2 ~ (b_sz, y2_sz)
        return self.Y2

    @staticmethod
    def _loss(Y_predicted, Y_true):  # mean cross-entropy over batch
        eps = 1e-12
        return -(Y_true * np.log(Y_predicted + eps)).sum() / Y_predicted.shape[0]

    @staticmethod
    def _dloss(Y_predicted, Y_true):  # softmax+CE: dL/dZ2 == (Y2 - Y_true)/N
        return (Y_predicted - Y_true) / Y_predicted.shape[0]

    def backward(self, Y_true, lr=1e-3):
        # E2 ~ (b_sz, y2_sz)
        self.E2 = self._dloss(self.Y2, Y_true) * self._df2(self.Z2)

        # E1 ~ (b_sz, y1_sz)
        self.E1 = self.E2 @ self.W2.T * self._df1(self.Z1)

        # dL_dW2 ~ (y1_sz, y2_sz)
        self.dL_dW2 = self.Y1.T @ self.E2
        # dL_db2 ~ (y2_sz,)   (bias gradient = error summed over the batch)
        self.dL_db2 = self.E2.sum(axis=0)

        # dL_dW1 ~ (x_sz, y1_sz)
        self.dL_dW1 = self.X.T @ self.E1
        # dL_db1 ~ (y1_sz,)   (bias gradient = error summed over the batch)
        self.dL_db1 = self.E1.sum(axis=0)

        self.W2 += -lr * self.dL_dW2
        self.b2 += -lr * self.dL_db2
        self.W1 += -lr * self.dL_dW1
        self.b1 += -lr * self.dL_db1

    @staticmethod
    def one_hot(labels, n_classes=10):  # integer labels (batch,) -> (batch, n_classes)
        Y = np.zeros((labels.shape[0], n_classes))
        Y[np.arange(labels.shape[0]), labels] = 1.0
        return Y

    def predict(self, X):  # class index with highest probability
        return self.forward(X).argmax(axis=1)


dnn_b = DigitsRecognizerNetB(x_sz=784, y1_sz=128, y2_sz=10)

W1: float64 (784, 128)
b1: float64 (128,)
W2: float64 (128, 10)
b2: float64 (10,)


# verifications

## Gradient checking: numerical vs analytical

**Intuition.** `backward()` computes the *analytical* gradient — a closed-form
$\partial L/\partial \theta$ derived with the chain rule. It's fast but easy to get subtly
wrong (a stray transpose, a missing activation derivative, a flipped sign, a sum over the
wrong axis). A gradient is just a **slope**, and we can measure a slope *independently* and
brute-force by nudging one parameter and watching the loss move. If the hand-derived
analytical gradient matches that measurement, backprop is almost certainly correct.

### The numerical gradient (finite differences)

By definition, the derivative w.r.t. a single parameter $\theta_i$ is

$$\frac{\partial L}{\partial \theta_i} = \lim_{\varepsilon \to 0} \frac{L(\theta_i+\varepsilon) - L(\theta_i-\varepsilon)}{2\varepsilon}.$$

We approximate it for a small $\varepsilon$ with the **central difference**, which is
symmetric and has error $O(\varepsilon^2)$ — far more accurate than the one-sided
forward difference $\big(L(\theta_i+\varepsilon)-L(\theta_i)\big)/\varepsilon$, whose error is only $O(\varepsilon)$:

$$g^{\text{num}}_i = \frac{L(\theta_i+\varepsilon) - L(\theta_i-\varepsilon)}{2\varepsilon}.$$

Each entry costs **two forward passes and no backward pass**. For a weight matrix with
millions of entries that's hopeless for training — but ideal as a *test*, especially if we
only spot-check a **random subset** of entries.

### Comparing the two

Use a scale-free **relative error** rather than a raw difference (a `1e-3` mismatch is fine
for huge gradients, catastrophic for tiny ones):

$$\text{rel} = \frac{|g^{\text{num}}_i - g^{\text{ana}}_i|}{|g^{\text{num}}_i| + |g^{\text{ana}}_i| + \text{tiny}}.$$

For `float64` central differences a correct gradient typically gives $\text{rel} \lesssim 10^{-7}$;
anything above ~$10^{-4}$ signals a bug.

### Pseudocode

```
analytical = backward(...)              # the gradient we want to trust
for i in sampled_parameter_indices:
    theta[i] += eps;   L_plus  = loss(forward())
    theta[i] -= 2*eps; L_minus = loss(forward())
    theta[i] += eps                     # restore exactly
    numerical_i = (L_plus - L_minus) / (2*eps)
    assert rel_err(numerical_i, analytical[i]) < tol
```

### Why *also* "check that it trains"

A gradient check validates **one step at one point**. It can still miss bugs that don't
surface there: a wrong learning-rate sign, an update written to the wrong buffer, stale
state carried across iterations, or exploding/vanishing dynamics. A cheap end-to-end smoke
test — run a few hundred SGD steps and confirm the loss actually goes **down** — catches
those. The two are complementary: the gradient check asks *"is the gradient right?"*, the
train check asks *"does using it actually learn?"*.

In [ ]:
from typing import Literal


def verify_network(
    net,
    *,
    task: Literal["classification", "regression"],
    batch_size=8,
    train_steps=300,
    lr=1e-2,
    n_grad_checks=30,  # random entries sampled per parameter (a full check is too slow)
    eps=1e-6,  # finite-difference step for the numerical gradient
    grad_tol=1e-4,  # max acceptable relative error between numerical & analytical grads
    seed=0,
):
    """Sanity-check an already-constructed network two complementary ways:
      (a) GRADIENT CHECK -- numerical gradient (central finite differences of the loss)
          vs analytical gradient (what backward() actually applies). Catches bugs in the
          backward math (transposes, signs, missing activation derivatives, wrong axes).
      (b) TRAIN CHECK    -- a short SGD loop must drive the loss DOWN. Catches bugs a
          one-point gradient check can miss (wrong lr sign, stale state, bad dynamics).

    Works for every net in this notebook: they share forward(X), backward(Y_true, lr),
    the staticmethod _loss(Y_pred, Y_true), and parameters named W1/W2 (plus b1/b2).
    """
    cls = type(net)
    x_sz = net.W1.shape[0]  # input size  (W1 is (x_sz, y1_sz))
    y2_sz = net.W2.shape[1]  # output size (W2 is (y1_sz, y2_sz))

    # one fixed (X, Y_true) batch for the chosen task
    rng = np.random.default_rng(seed)
    X = rng.standard_normal((batch_size, x_sz))
    if task == "classification":  # one-hot targets so cross-entropy is well defined
        labels = rng.integers(0, y2_sz, size=batch_size)
        Y_true = np.zeros((batch_size, y2_sz))
        Y_true[np.arange(batch_size), labels] = 1.0
    else:  # real-valued targets for the squared-error loss
        Y_true = rng.standard_normal((batch_size, y2_sz))

    loss = lambda: cls._loss(net.forward(X), Y_true)  # scalar loss at current params
    # parameter arrays the net exposes (biases only on the *B variants)
    params = [
        p
        for p in ("W1", "b1", "W2", "b2")
        if isinstance(getattr(net, p, None), np.ndarray)
    ]

    # ---------- (a) GRADIENT CHECK ----------
    # Read the analytical gradient straight from the update: every class does
    #   P <- P - lr * grad,  so ONE step with lr=1 makes the amount each parameter
    # moved equal its gradient (this avoids depending on internal grad-buffer details).
    net.forward(X)
    before = {p: getattr(net, p).copy() for p in params}
    net.backward(Y_true, lr=1.0)
    analytic = {p: before[p] - getattr(net, p) for p in params}
    for p in params:
        getattr(net, p)[:] = before[p]  # put the parameters back

    # numerical gradient on a random subset of entries; compare via relative error
    worst_rel = 0.0
    for p in params:
        flat = getattr(net, p).reshape(
            -1
        )  # view -> writing flat[i] edits the real parameter
        a_flat = analytic[p].reshape(-1)
        for i in rng.choice(
            flat.size, size=min(n_grad_checks, flat.size), replace=False
        ):
            o = flat[i]
            flat[i] = o + eps
            lp = loss()
            flat[i] = o - eps
            lm = loss()
            flat[i] = o  # restore exactly
            num = (lp - lm) / (2 * eps)
            worst_rel = max(
                worst_rel, abs(num - a_flat[i]) / (abs(num) + abs(a_flat[i]) + 1e-12)
            )

    # ---------- (b) TRAIN CHECK ----------
    # the gradient check left the parameters at their initial values, so just train now
    L0 = loss()
    for _ in range(train_steps):
        net.forward(X)
        net.backward(Y_true, lr=lr)
    L1 = loss()

    # ---------- report ----------
    print(f"{cls.__name__}  (params: {params}, task: {task})")
    print(
        f"  (a) gradient check: worst rel err = {worst_rel:.2e}  -> {'PASS' if worst_rel < grad_tol else 'FAIL'}  (tol {grad_tol:.0e})"
    )
    print(
        f"  (b) train check   : loss {L0:.4f} -> {L1:.4f}  -> {'PASS' if L1 < L0 else 'FAIL'}"
    )
    if task == "classification" and hasattr(net, "predict"):
        print(
            f"      train accuracy = {(net.predict(X) == Y_true.argmax(1)).mean():.2f}"
        )


verify_network(Net(x_sz=5, y1_sz=16, y2_sz=2), task="regression")
verify_network(NetB(x_sz=5, y1_sz=16, y2_sz=2), task="regression")
verify_network(NetFixedMem(b_sz=8, x_sz=5, y1_sz=16, y2_sz=2), task="regression")
verify_network(NetFixedMemB(b_sz=8, x_sz=5, y1_sz=16, y2_sz=2), task="regression")
verify_network(DigitsRecognizerNet(x_sz=5, y1_sz=16, y2_sz=2), task="classification")
verify_network(DigitsRecognizerNetB(x_sz=5, y1_sz=16, y2_sz=2), task="classification")

W1: float64 (5, 16)
W2: float64 (16, 2)
Net  (params: ['W1', 'W2'], task: regression)
  (a) gradient check: worst rel err = 3.16e-08  -> PASS  (tol 1e-04)
  (b) train check   : loss 20.2715 -> 0.1278  -> PASS
W1: float64 (5, 16)
b1: float64 (16,)
W2: float64 (16, 2)
b2: float64 (2,)
NetB  (params: ['W1', 'b1', 'W2', 'b2'], task: regression)
  (a) gradient check: worst rel err = 1.25e-08  -> PASS  (tol 1e-04)
  (b) train check   : loss 23.4081 -> 0.0107  -> PASS
W1: float64 (5, 16)
W2: float64 (16, 2)
X: float64 (8, 5)
Z1: float64 (8, 16)
Y1: float64 (8, 16)
Z2: float64 (8, 2)
Y2: float64 (8, 2)
E2: float64 (8, 2)
E1: float64 (8, 16)
D1: float64 (8, 16)
D2: float64 (8, 2)
dL_dW1: float64 (5, 16)
dL_dW2: float64 (16, 2)
NetFixedMem  (params: ['W1', 'W2'], task: regression)
  (a) gradient check: worst rel err = 1.44e-08  -> PASS  (tol 1e-04)
  (b) train check   : loss 15.3465 -> 0.1990  -> PASS
W1: float64 (5, 16)
b1: float64 (16,)
W2: float64 (16, 2)
b2: float64 (2,)
X: float64 (8, 5)
Z1

# task example: digits recognizing

In [ ]:
# ...

# [EOF]